# Bronze → Silver · V-Credit

## Objetivo
Aplicar as regras de negócio de cada base e gravar tabelas tipadas, padronizadas e confiáveis na camada Silver.

## Ordem de execução
1. Bases de referência: canais, motivos, atendentes e clientes
2. Chamados_Hora, chamados, custos e pesquisa de satisfação, que dependem das bases de referência

## Log de qualidade
Cada regra registra quantas linhas afetou. O log é gravado em `tb_log_qualidade` ao final do notebook e é a base da leitura de confiabilidade dos dados.

In [0]:
from pyspark.sql import functions as F

catalogo = "vcredit"
bronze = f"{catalogo}.bronze"
silver = f"{catalogo}.silver"

# Datas fora do padrão viram NULL em vez de gerar erro de versão do parser
spark.conf.set("spark.sql.legacy.timeParserPolicy", "CORRECTED")

# Dicionário por (base, regra): reexecutar uma célula atualiza o registro em vez de duplicar
log_qualidade = {}


def registrar(base, regra, linhas_afetadas, total_linhas):
    log_qualidade[(base, regra)] = (int(linhas_afetadas), int(total_linhas))


def chave_texto(coluna):
    # Minúsculas, sem acento e sem espaços extras: base para comparar textos digitados de formas diferentes
    com_acento = "áàâãäéèêëíìîïóòôõöúùûüç"
    sem_acento = "aaaaaeeeeiiiiooooouuuuc"
    return F.trim(F.regexp_replace(F.translate(F.lower(coluna), com_acento, sem_acento), r"\s+", " "))


def de_para(dicionario):
    # Converte um dicionário Python em um mapa do Spark para traduzir valores coluna a coluna
    return F.create_map([F.lit(valor) for par in dicionario.items() for valor in par])


def gravar_silver(df, tabela):
    # Overwrite garante a idempotência: reexecutar substitui a tabela em vez de duplicar
    df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{silver}.{tabela}")
    print(f"{silver}.{tabela}: {spark.table(f'{silver}.{tabela}').count()} linhas")

## 1. tb_canais

- **Nome:** padronizado para URA, Chatbot, Atendimento Inicial e Atendimento Especializado
- **Tipo:** padronizado para Autosserviço e Humano
- **Nível:** nova coluna com a etapa do funil (Autosserviço, N1 ou N2)
- **Duplicatas:** removidas por `id_canal`, como proteção para cargas futuras
- Um canal fora do de-para interrompe o pipeline, pois indica um canal novo na origem

In [0]:
# De-para do nome normalizado (só letras) para o nome padronizado
nomes_canais = {
    "ura": "URA",
    "chatbot": "Chatbot",
    "atendimentoinicial": "Atendimento Inicial",
    "atendimentoespecializado": "Atendimento Especializado",
}
tipos_canais = {"autosservico": "Autosserviço", "humano": "Humano"}

# Etapa do funil de cada canal
niveis_canais = {
    "URA": "Autosserviço",
    "Chatbot": "Autosserviço",
    "Atendimento Inicial": "N1",
    "Atendimento Especializado": "N2",
}

df_bronze = spark.table(f"{bronze}.tb_canais")
total = df_bronze.count()

# Duplicatas: remoção por id_canal como proteção para cargas futuras
df_bronze = df_bronze.dropDuplicates(["id_canal"])
registrar("tb_canais", "Duplicatas por id_canal removidas", total - df_bronze.count(), total)

df_canais = (
    df_bronze
    .select(
        # Tipagem: id como inteiro
        F.col("id_canal").cast("int").alias("id_canal"),
        # Nome: mantém só as letras do texto normalizado e aplica o de-para
        de_para(nomes_canais)[F.regexp_replace(chave_texto(F.col("nome_canal")), "[^a-z]", "")].alias("nome_canal"),
        # Tipo: "Autosservico" ganha o acento e fica Autosserviço
        de_para(tipos_canais)[chave_texto(F.col("tipo_canal"))].alias("tipo_canal"),
        # Status: em minúsculas
        F.lower(F.trim(F.col("status_canal"))).alias("status_canal"),
    )
    # Nível: nova coluna com a etapa do funil
    .withColumn("nivel_canal", de_para(niveis_canais)[F.col("nome_canal")])
)

# Canal fora do de-para indica um canal novo na origem: o pipeline para em vez de seguir com NULL
if df_canais.filter(F.col("nome_canal").isNull() | F.col("tipo_canal").isNull()).count() > 0:
    raise ValueError("tb_canais possui canal ou tipo fora do de-para.")

gravar_silver(df_canais, "tb_canais")

vcredit.silver.tb_canais: 4 linhas


## 2. tb_motivos

- **Categoria:** grafia padronizada (Financeiro, Cartão, Cadastral, Atendimento, Benefícios). Os 3 motivos sem categoria são classificados pelos critérios do case: Contratação de cartão adicional em Cartão, Transferência de agência em Cadastral e Cancelamento de conta em Financeiro
- **Criticidade:** padronizada para Baixa, Média e Alta
- **chave_motivo:** nome do motivo em minúsculas, sem acento e sem espaços extras, usado para ligar o texto livre dos chamados ao `id_motivo`
- **Duplicatas:** removidas por `id_motivo`

In [0]:
# De-para da categoria normalizada para a grafia padronizada
categorias = {
    "financeiro": "Financeiro",
    "cartao": "Cartão",
    "cadastral": "Cadastral",
    "atendimento": "Atendimento",
    "beneficios": "Benefícios",
}

# Motivos sem categoria na origem, classificados pelos critérios do case
categorias_sem_origem = {
    "contratacao de cartao adicional": "Cartão",
    "transferencia de agencia": "Cadastral",
    "cancelamento de conta": "Financeiro",
}
criticidades = {"baixa": "Baixa", "media": "Média", "alta": "Alta"}

df_bronze = spark.table(f"{bronze}.tb_motivos")
total = df_bronze.count()

# Duplicatas: remoção por id_motivo como proteção para cargas futuras
df_bronze = df_bronze.dropDuplicates(["id_motivo"])
registrar("tb_motivos", "Duplicatas por id_motivo removidas", total - df_bronze.count(), total)

# Chave de associação: nome normalizado, usado para ligar o texto livre dos chamados ao id_motivo
df_motivos = df_bronze.withColumn("chave_motivo", chave_texto(F.col("nome_motivo")))

# Registro no log das inconsistências encontradas antes do tratamento
sem_categoria = F.col("categoria").isNull() | (F.trim(F.col("categoria")) == "")
registrar("tb_motivos", "Categoria vazia classificada pelos critérios do case", df_motivos.filter(sem_categoria).count(), total)
registrar(
    "tb_motivos",
    "Categoria com grafia fora do padrão",
    df_motivos.filter(~sem_categoria & ~F.col("categoria").isin(list(categorias.values()))).count(),
    total,
)
registrar(
    "tb_motivos",
    "Criticidade com grafia fora do padrão",
    df_motivos.filter(~F.col("criticidade").isin(list(criticidades.values()))).count(),
    total,
)

df_motivos = df_motivos.select(
    # Tipagem: id como inteiro
    F.col("id_motivo").cast("int").alias("id_motivo"),
    F.trim(F.col("nome_motivo")).alias("nome_motivo"),
    # Categoria: vazia é classificada pelo motivo; preenchida tem a grafia padronizada
    F.when(sem_categoria, de_para(categorias_sem_origem)[F.col("chave_motivo")])
    .otherwise(de_para(categorias)[chave_texto(F.col("categoria"))])
    .alias("categoria"),
    # Criticidade: padronizada para Baixa, Média e Alta
    de_para(criticidades)[chave_texto(F.col("criticidade"))].alias("criticidade"),
    "chave_motivo",
)

# Categoria ou criticidade fora do de-para interrompe o pipeline
if df_motivos.filter(F.col("categoria").isNull() | F.col("criticidade").isNull()).count() > 0:
    raise ValueError("tb_motivos possui categoria ou criticidade fora do de-para.")

gravar_silver(df_motivos, "tb_motivos")

vcredit.silver.tb_motivos: 14 linhas


## 3. tb_atendentes

- **Nível:** padronizado para 1 e 2 a partir das variações (1, Nível 1, N2, Nível 2)
- **Datas:** admissão e desligamento convertidas de dd/MM/aaaa; desligamento vazio vira NULL
- **Status:** recalculado pela data de desligamento (com data, inativo; sem data, ativo), com registro no log se divergir da origem
- **Duplicatas:** removidas por `id_atendente`

In [0]:
df_bronze = spark.table(f"{bronze}.tb_atendentes")
total = df_bronze.count()

# Duplicatas: remoção por id_atendente como proteção para cargas futuras
df_bronze = df_bronze.dropDuplicates(["id_atendente"])
registrar("tb_atendentes", "Duplicatas por id_atendente removidas", total - df_bronze.count(), total)

# Registro no log dos níveis escritos fora do padrão numérico
registrar(
    "tb_atendentes",
    "Nível de atendimento fora do padrão (Nível 1, N2...)",
    df_bronze.filter(~F.trim(F.col("nivel_atendimento")).rlike(r"^[12]$")).count(),
    total,
)

df_atendentes = (
    df_bronze
    .select(
        # Tipagem: id como inteiro
        F.col("id_atendente").cast("int").alias("id_atendente"),
        F.trim(F.col("nome_atendente")).alias("nome_atendente"),
        # Nível: extrai o dígito de qualquer variação (1, Nível 1, N2...)
        F.regexp_extract(F.col("nivel_atendimento"), r"([12])", 1).cast("int").alias("nivel_atendimento"),
        # Datas: dd/MM/aaaa para date; desligamento vazio vira NULL
        F.to_date(F.try_to_timestamp(F.trim(F.col("data_admissao")), F.lit("dd/MM/yyyy"))).alias("data_admissao"),
        F.to_date(F.try_to_timestamp(F.trim(F.col("data_desligamento")), F.lit("dd/MM/yyyy"))).alias("data_desligamento"),
        F.lower(F.trim(F.col("status_atendente"))).alias("status_origem"),
    )
    # Status: recalculado pela data de desligamento
    .withColumn("status_atendente", F.when(F.col("data_desligamento").isNotNull(), "inativo").otherwise("ativo"))
)

# Registro no log de status da origem que não batem com a data de desligamento
registrar(
    "tb_atendentes",
    "Status divergente da data de desligamento",
    df_atendentes.filter(F.col("status_origem") != F.col("status_atendente")).count(),
    total,
)

df_atendentes = df_atendentes.drop("status_origem")

# Nível ou data de admissão inválidos interrompem o pipeline
if df_atendentes.filter(F.col("nivel_atendimento").isNull() | F.col("data_admissao").isNull()).count() > 0:
    raise ValueError("tb_atendentes possui nível ou data de admissão inválidos.")

gravar_silver(df_atendentes, "tb_atendentes")

vcredit.silver.tb_atendentes: 25 linhas


## 4. tb_clientes

- **Região:** espaços removidos e associada por de-para às 5 regiões (Norte, Nordeste, Centro-Oeste, Sudeste e Sul)
- **Data de cadastro:** convertida a partir dos formatos dd/MM/aaaa e aaaa-MM-dd
- **Idade:** válida entre 18 e 100 anos; fora disso vira NULL (a base tem idades 0 e acima de 120)
- **Faixa etária:** nova coluna (18 a 29, 30 a 44, 45 a 59, 60 a 74, 75 ou mais, Não informado)
- **E-mail:** sem "@" vira NULL
- **Dia de vencimento:** convertido para inteiro
- **id_cliente:** mantido como texto de 11 dígitos, preservando zeros à esquerda

In [0]:
# De-para da região normalizada; evita o initcap, que geraria "Centro-oeste"
regioes = {
    "norte": "Norte",
    "nordeste": "Nordeste",
    "centro-oeste": "Centro-Oeste",
    "sudeste": "Sudeste",
    "sul": "Sul",
}

df_bronze = spark.table(f"{bronze}.tb_clientes")
total = df_bronze.count()

# Duplicatas: remoção por id_cliente como proteção para cargas futuras
df_bronze = df_bronze.dropDuplicates(["id_cliente"])
registrar("tb_clientes", "Duplicatas por id_cliente removidas", total - df_bronze.count(), total)

# Idade convertida só quando o texto é numérico, para não gerar erro de conversão
idade_numero = F.when(F.trim(F.col("idade")).rlike(r"^\d+$"), F.trim(F.col("idade")).cast("int"))
data_iso = F.trim(F.col("data_cadastro")).rlike(r"^\d{4}-\d{2}-\d{2}$")

# Registro no log das inconsistências encontradas antes do tratamento
registrar(
    "tb_clientes",
    "Região com grafia fora do padrão (maiúsculas ou espaços)",
    df_bronze.filter(~F.col("regiao").isin(list(regioes.values()))).count(),
    total,
)
registrar("tb_clientes", "Data de cadastro no formato aaaa-MM-dd", df_bronze.filter(data_iso).count(), total)
registrar("tb_clientes", "Idade ausente", df_bronze.filter(idade_numero.isNull()).count(), total)
registrar("tb_clientes", "Idade fora de 18 a 100 convertida para NULL", df_bronze.filter(~idade_numero.between(18, 100)).count(), total)
registrar("tb_clientes", "E-mail sem @ convertido para NULL", df_bronze.filter(~F.col("email").contains("@")).count(), total)

# Idade: válida entre 18 e 100; fora disso vira NULL
idade = F.when(idade_numero.between(18, 100), idade_numero)

df_clientes = (
    df_bronze
    .select(
        # Chave: id_cliente como texto, preservando zeros à esquerda
        F.trim(F.col("id_cliente")).alias("id_cliente"),
        F.trim(F.col("nome")).alias("nome"),
        # E-mail: sem "@" é inválido e vira NULL
        F.when(F.col("email").contains("@"), F.trim(F.col("email"))).alias("email"),
        # Região: espaços removidos e de-para para as 5 regiões
        de_para(regioes)[chave_texto(F.col("regiao"))].alias("regiao"),
        idade.alias("idade"),
        # Faixa etária: nova coluna; idade NULL fica como Não informado
        F.when(idade < 30, "18 a 29")
        .when(idade < 45, "30 a 44")
        .when(idade < 60, "45 a 59")
        .when(idade < 75, "60 a 74")
        .when(idade.isNotNull(), "75 ou mais")
        .otherwise("Não informado")
        .alias("faixa_etaria"),
        # Data de cadastro: tenta dd/MM/aaaa e, se não casar, aaaa-MM-dd
        F.coalesce(
            F.to_date(F.try_to_timestamp(F.trim(F.col("data_cadastro")), F.lit("dd/MM/yyyy"))),
            F.to_date(F.try_to_timestamp(F.trim(F.col("data_cadastro")), F.lit("yyyy-MM-dd"))),
        ).alias("data_cadastro"),
        # Dia de vencimento: convertido para inteiro
        F.col("dia_vencimento_fatura").cast("int").alias("dia_vencimento_fatura"),
    )
)

# id sem 11 dígitos, região fora do de-para ou data inválida interrompem o pipeline
if df_clientes.filter(
    ~F.col("id_cliente").rlike(r"^\d{11}$") | F.col("regiao").isNull() | F.col("data_cadastro").isNull()
).count() > 0:
    raise ValueError("tb_clientes possui id, região ou data de cadastro inválidos.")

gravar_silver(df_clientes, "tb_clientes")

vcredit.silver.tb_clientes: 47705 linhas


## Log de qualidade

Consolida as regras registradas ao longo do notebook em `tb_log_qualidade`.

(deixar essa célula como ultima)

In [0]:
# Consolida as regras registradas no notebook; deve ser sempre a última célula
df_log = (
    spark.createDataFrame(
        [(base, regra, afetadas, total) for (base, regra), (afetadas, total) in log_qualidade.items()],
        "base string, regra string, linhas_afetadas int, total_linhas int",
    )
    # Percentual da base afetado por cada regra
    .withColumn("percentual", F.round(F.col("linhas_afetadas") / F.col("total_linhas") * 100, 2))
    .withColumn("data_execucao", F.current_timestamp())
)
1
gravar_silver(df_log, "tb_log_qualidade")
display(df_log)

vcredit.silver.tb_log_qualidade: 14 linhas


base,regra,linhas_afetadas,total_linhas,percentual,data_execucao
tb_canais,Duplicatas por id_canal removidas,0,4,0.0,2026-10-26T17:50:11.503Z
tb_motivos,Duplicatas por id_motivo removidas,0,14,0.0,2026-10-26T17:50:11.503Z
tb_motivos,Categoria vazia classificada pelos critérios do case,3,14,21.43,2026-10-26T17:50:11.503Z
tb_motivos,Categoria com grafia fora do padrão,4,14,28.57,2026-10-26T17:50:11.503Z
tb_motivos,Criticidade com grafia fora do padrão,11,14,78.57,2026-10-26T17:50:11.503Z
tb_atendentes,Duplicatas por id_atendente removidas,0,25,0.0,2026-10-26T17:50:11.503Z
tb_atendentes,"Nível de atendimento fora do padrão (Nível 1, N2...)",8,25,32.0,2026-10-26T17:50:11.503Z
tb_atendentes,Status divergente da data de desligamento,0,25,0.0,2026-10-26T17:50:11.503Z
tb_clientes,Duplicatas por id_cliente removidas,0,47705,0.0,2026-10-26T17:50:11.503Z
tb_clientes,Região com grafia fora do padrão (maiúsculas ou espaços),5168,47705,10.83,2026-10-26T17:50:11.503Z
